# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR⁲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn to load data defined by a Croissant schema, inspect record sets and fields via their `@id`s, extract data into DataFrames, and perform typical preprocessing/EDA steps.

### Dataset Source
The dataset is specified by a Croissant schema URL.

- Dataset: *Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution* ([FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p))
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)

## 2. Data Overview
Let's inspect the list of record sets (`@id`), and their fields and columns with their `@id`s. This gives an overview of the structure for further referencing.

In [ ]:
# List available record sets, and fields in each, referencing by @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets are present in the Croissant metadata.")
else:
    print("Available record sets and their fields (all referenced by @id):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        if 'field' in rs:
            field_list = rs['field']
            if isinstance(field_list, dict):
                fields = [field_list]
            else:
                fields = field_list
            for field in fields:
                print(f"  Field @id: {field['@id']}")
        if 'column' in rs:
            column_list = rs['column']
            if isinstance(column_list, dict):
                columns = [column_list]
            else:
                columns = column_list
            for col in columns:
                print(f"  Column @id: {col['@id']}")
        print()

### Show a sample record: 
Now, try loading and displaying one record from each record set. This helps explore available fields and the structure of the records.

In [ ]:
# List and display a sample record from each record set using their @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f'\nSample record for record set @id: {rs_id}')
    records_iter = dataset.records(record_set=rs_id)
    try:
        record = next(records_iter)
        print(json.dumps(record, indent=2))
    except StopIteration:
        print('No records found!')
    except Exception as e:
        print(f'Error loading records: {e}')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. We refer to record sets and fields using their `@id`.

In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
    else:
        print("  No records found.")

For demonstration, we'll now select the **main tabular record set** (containing the clinical variables data). You may need to adjust the record set `@id` below if it's different. You can review the printouts above for the relevant `@id`.

In [ ]:
# Choose the main record set id. Adjust as needed based on overview above.

# For illustration, let's try to find the first tabular record set:
main_rs_id = None
for rs in record_sets:
    if 'field' in rs or 'column' in rs:
        main_rs_id = rs['@id']
        break

if main_rs_id is not None and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    print(f"Loaded DataFrame for record set @id {main_rs_id}:\n", df.head())
else:
    print("No suitable tabular record set loaded. Please check available IDs.")

## 4. Exploratory Data Analysis (EDA)
We'll now apply basic data processing: filtering numerically, normalizing, and grouping. Replace the field `@id`s below with those matching numeric and groupable fields from your DataFrame discovery.

In [ ]:
# Set your numeric field and grouping field by @id (string column names in DataFrame)
# These should correspond to the @id used as columns in df. For example:
# numeric_field_id = 'http://mlcommons.org/croissant/field/age'     # example only
# group_field_id = 'http://mlcommons.org/croissant/field/sex'

numeric_field_id = None
group_field_id = None

if main_rs_id is not None and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    print("Available columns (as @id):", df.columns.tolist())
    # Try to guess some candidates for numeric and groupable fields
    # Look for a likely numeric field
    for col in df.columns:
        if (df[col].dtype.kind in 'iufc' and col.lower().find('age') >= 0):
            numeric_field_id = col
        if (df[col].dtype == 'O' and col.lower().find('sex') >= 0):
            group_field_id = col

    if numeric_field_id is None:
        # Fallback: try any float/int column
        numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'iufc']
        numeric_field_id = numeric_candidates[0] if numeric_candidates else None

    if group_field_id is None:
        group_candidates = [col for col in df.columns if df[col].dtype == 'O']
        group_field_id = group_candidates[0] if group_candidates else None

    if numeric_field_id is not None:
        print(f"\nUsing numeric field for filtering and normalization: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            # Only group numeric columns
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df)
    else:
        print("No numeric field detected for EDA.")
else:
    print("No loaded tabular DataFrame. Skipping EDA.")

## 5. Visualization
Visualize numeric field distributions and relationships. Reference fields by their `@id` columns.

In [ ]:
import matplotlib.pyplot as plt

if main_rs_id is not None and main_rs_id in dataframes and numeric_field_id is not None:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].plot(kind='hist', bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(7, 5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
- We have programmatically loaded metadata and tabular clinical records referenced by their Croissant `@id`s via the FAIR² Croissant schema and the `mlcroissant` library.
- The structure of the dataset, record set, and fields can be fully accessed and referenced by their `@id`, supporting reproducible data science workflows.
- We extracted clinical data into DataFrames, performed standard filtering, normalization, and grouping, and visualized numeric distributions and group relationships.

**Next steps:** Explore remaining fields, perform statistical analysis or modeling, and check Croissant documentation for advanced data integration.